In [1]:
# Install all required packages
!pip install -q rank-bm25 sentence-transformers groq google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.2 MB/s eta 0:00:00


In [9]:
import os
import re
import math
import numpy as np
from typing import List, Dict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

GROQ_API_KEY    = ""

groq_client = Groq(api_key=GROQ_API_KEY)

print("All imports successful. Clients initialised.")

All imports successful. Clients initialised.


In [5]:
CORPUS = [
    # Doc 0
    "The attention mechanism allows a model to dynamically weight different "
    "parts of the input sequence when producing each output token, enabling "
    "it to capture long-range dependencies without recurrence.",

    # Doc 1
    "Self-attention computes a weighted sum of value vectors, where the "
    "weights are derived from the dot product of query and key vectors, "
    "scaled by the square root of the key dimension.",

    # Doc 2
    "Positional encodings are added to token embeddings in transformer "
    "models so that the architecture can encode the order of tokens, since "
    "self-attention itself is permutation-invariant.",

    # Doc 3 — gradient descent
    "Stochastic gradient descent updates model weights by computing the "
    "gradient of the loss on a mini-batch and stepping in the negative "
    "gradient direction, balancing speed and stability.",

    # Doc 4 — regularisation
    "Dropout is a regularization technique that randomly sets neuron "
    "activations to zero during training, forcing the network to learn "
    "redundant representations and reducing overfitting.",

    # Doc 5 — learning rate schedules
    "Cosine annealing gradually reduces the learning rate following a "
    "cosine curve, often combined with warm restarts to escape local minima "
    "and converge to flatter, more generalizable solutions.",

    # Doc 6
    "Vaswani et al. (2017) introduced the Transformer architecture in "
    "'Attention Is All You Need', replacing recurrence with multi-head "
    "self-attention and achieving state-of-the-art BLEU scores on WMT.",

    # Doc 7 — backpropagation
    "Backpropagation applies the chain rule to compute gradients of the "
    "loss with respect to each parameter, propagating error signals "
    "backwards through every layer of a neural network.",

    # Doc 8 — embeddings
    "Word embeddings such as Word2Vec and GloVe represent tokens as dense "
    "vectors in a continuous space, capturing semantic similarity so that "
    "analogous words cluster together geometrically.",

    # Doc 9 — CNNs
    "Convolutional neural networks apply learned filters across local "
    "receptive fields to extract spatial hierarchies of features, making "
    "them highly effective for image recognition tasks.",

    # Doc 10 — BERT
    "BERT (Bidirectional Encoder Representations from Transformers) "
    "pre-trains a deep transformer using masked language modelling and "
    "next-sentence prediction, producing context-aware token embeddings.",

    # Doc 11 — reinforcement learning
    "Reinforcement learning trains an agent to maximise cumulative reward "
    "by interacting with an environment; policy gradient methods directly "
    "optimise the expected return using sampled trajectories.",

    # Doc 12 — optimisers (AdamW)
    "AdamW decouples weight decay from the adaptive gradient update in "
    "Adam, preventing the optimiser from inadvertently scaling down the "
    "regularisation effect and improving generalisation in transformers.",

    # Doc 13 — batch normalisation
    "Batch normalisation standardises layer inputs across the mini-batch "
    "dimension, reducing internal covariate shift and allowing the use of "
    "higher learning rates without destabilising training.",

    # Doc 14 — transfer learning
    "Transfer learning fine-tunes a model pre-trained on a large corpus "
    "for a downstream task, requiring far fewer labelled examples than "
    "training from scratch and typically yielding better generalisation.",
]

In [7]:
class HybridRetriever:

    def __init__(self, corpus: List[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        tokenised = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenised)
        print("BM25 index built.")

        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.corpus_embeddings = self.sbert.encode(
            corpus, convert_to_numpy=True, show_progress_bar=True
        )
        print(f"SBERT index built. Embedding shape: {self.corpus_embeddings.shape}")

    def _bm25_ranked(self, query: str) -> List[int]:
        """Return document indices sorted by BM25 score, best first."""
        tokenised_query = query.lower().split()
        scores = self.bm25.get_scores(tokenised_query)
        return list(np.argsort(scores)[::-1])

    def _sbert_ranked(self, query: str) -> List[int]:
        """Return document indices sorted by cosine similarity, best first."""
        query_emb = self.sbert.encode([query], convert_to_numpy=True)
        sims = cosine_similarity(query_emb, self.corpus_embeddings)[0]
        return list(np.argsort(sims)[::-1])

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        bm25_order  = self._bm25_ranked(query)
        sbert_order = self._sbert_ranked(query)

        bm25_rank  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}
        sbert_rank = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_order)}

        rrf_scores = {}
        for doc_id in range(len(self.corpus)):
            rrf_scores[doc_id] = (
                1.0 / (self.k + bm25_rank[doc_id])
                + 1.0 / (self.k + sbert_rank[doc_id])
            )

        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, score in sorted_docs[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(score, 6),
                "bm25_rank":  bm25_rank[doc_id],
                "sbert_rank": sbert_rank[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results


retriever = HybridRetriever(CORPUS, k=60)

BM25 index built.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SBERT index built. Embedding shape: (15, 384)


In [20]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded.")


def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    if not candidates:
        return []

    # Build (query, document) pairs for the cross-encoder
    pairs = [(query, c["text"]) for c in candidates]

    # Score all pairs in one batch
    ce_scores = cross_encoder.predict(pairs)  # returns a numpy array

    # Attach scores and sort
    for candidate, score in zip(candidates, ce_scores):
        candidate["ce_score"] = round(float(score), 4)

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]


print("\n── Re-ranker demo ──")
_candidates = retriever.retrieve("what is self-attention?", top_k=5)
_reranked   = rerank("what is self-attention?", _candidates, top_k=3)
for r in _reranked:
    print(f"  Doc {r['doc_id']:02d} | CE={r['ce_score']:+.4f} | {r['text'][:70]}...")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cross-encoder loaded.

── Re-ranker demo ──
  Doc 02 | CE=+3.7166 | Positional encodings are added to token embeddings in transformer mode...
  Doc 06 | CE=+0.6917 | Vaswani et al. (2017) introduced the Transformer architecture in 'Atte...
  Doc 04 | CE=-10.2352 | Dropout is a regularization technique that randomly sets neuron activa...


In [11]:
def hyde_expand(user_query: str) -> str:
    prompt = (
        "You are a technical AI/ML textbook. "
        "Write a concise, factual 2-3 sentence answer to the following question "
        "using precise technical vocabulary. "
        "Do NOT say 'I' or start with 'The answer is'. "
        "Write as if it is a paragraph from a textbook.\n\n"
        f"Question: {user_query}"
    )

    # temperature=0.0 for deterministic, factual output
    # response = gemini_model.generate_content(
    #     prompt,
    #     generation_config=genai.GenerationConfig(temperature=0.0)
    # )
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant", # Using Groq model for HyDE expansion
        messages=[
            {"role": "user",   "content": prompt},
        ],
        temperature=0.0, # Equivalent to deterministic output
        max_tokens=300,  # Limiting response length for concise output
    )
    return response.choices[0].message.content.strip()

_q    = "how do transformers encode meaning?"
_hyde = hyde_expand(_q)
print(f"Original query : {_q}")
print(f"HyDE expansion :\n  {_hyde}")

Original query : how do transformers encode meaning?
HyDE expansion :
  Transformers encode meaning through self-attention mechanisms, which enable parallelized computation of weighted sums of input sequences. This is achieved by computing attention weights based on the similarity between input tokens, allowing the model to selectively focus on relevant information and weigh its importance. The resulting contextualized representations are then used to compute output vectors, capturing complex semantic relationships and dependencies within the input sequence.


In [19]:
def advanced_rag(user_query: str, verbose: bool = True) -> str:

    if verbose:
        print("\n" + "═" * 60)
        print(f"Query: {user_query}")
        print("─" * 60)
        print("[Step 1] Generating HyDE expansion...")

    hypothetical_doc = hyde_expand(user_query)

    if verbose:
        print(f"  HyDE doc: {hypothetical_doc[:120]}...")

    if verbose:
        print("[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...")

    candidates = retriever.retrieve(hypothetical_doc, top_k=5)

    if verbose:
        for c in candidates:
            print(f"  Doc {c['doc_id']:02d} | RRF={c['rrf_score']:.5f} "
                  f"| BM25={c['bm25_rank']:2d} | SBERT={c['sbert_rank']:2d} "
                  f"| {c['text'][:55]}...")

    if verbose:
        print("[Step 3] Cross-encoder re-ranking (original query)...")

    top_docs = rerank(user_query, candidates, top_k=3)

    if verbose:
        for d in top_docs:
            print(f"  Doc {d['doc_id']:02d} | CE={d['ce_score']:+.4f} | {d['text'][:65]}...")

    if verbose:
        print("[Step 4] Generating final answer with Groq...")

    context = "\n\n".join(
        f"[Doc {d['doc_id']}] {d['text']}" for d in top_docs
    )

    system_prompt = (
        "You are a knowledgeable university AI/ML assistant. "
        "Answer the student's question using ONLY the provided context documents. "
        "Be concise (2-4 sentences), accurate, and use correct technical terminology. "
        "If the context does not contain enough information, say so honestly."
    )

    user_prompt = (
        f"Context:\n{context}\n\n"
        f"Student question: {user_query}\n\n"
        "Answer:"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=0.2,
        max_tokens=300,
    )

    answer = response.choices[0].message.content.strip()

    if verbose:
        print("\n[Answer]")
        print(f"  {answer}")
        print("═" * 60)

    return answer


print("advanced_rag() function defined.")

advanced_rag() function defined.


In [14]:
# Test query 1
from typing import List, Dict
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    if not candidates:
        return []
    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)
    for candidate, score in zip(candidates, ce_scores):
        candidate["ce_score"] = round(float(score), 4)
    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]

answer_q1 = advanced_rag("how do transformers encode meaning?")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


════════════════════════════════════════════════════════════
Query: how do transformers encode meaning?
────────────────────────────────────────────────────────────
[Step 1] Generating HyDE expansion...
  HyDE doc: Transformers encode meaning through self-attention mechanisms, which enable parallelized computation of weighted sums of...
[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  Doc 00 | RRF=0.03279 | BM25= 1 | SBERT= 1 | The attention mechanism allows a model to dynamically w...
  Doc 01 | RRF=0.03151 | BM25= 2 | SBERT= 5 | Self-attention computes a weighted sum of value vectors...
  Doc 02 | RRF=0.03150 | BM25= 4 | SBERT= 3 | Positional encodings are added to token embeddings in t...
  Doc 06 | RRF=0.03101 | BM25= 5 | SBERT= 4 | Vaswani et al. (2017) introduced the Transformer archit...
  Doc 03 | RRF=0.02996 | BM25= 3 | SBERT=11 | Stochastic gradient descent updates model weights by co...
[Step 3] Cross-encoder re-ranking (original query)...
  Doc 02 | CE=-2.9917 | Positio

In [15]:
# Test query 2
answer_q2 = advanced_rag("optimization techniques for training")


════════════════════════════════════════════════════════════
Query: optimization techniques for training
────────────────────────────────────────────────────────────
[Step 1] Generating HyDE expansion...
  HyDE doc: Optimization techniques for training machine learning models involve the use of algorithms to minimize the loss function...
[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  Doc 03 | RRF=0.03279 | BM25= 1 | SBERT= 1 | Stochastic gradient descent updates model weights by co...
  Doc 12 | RRF=0.03200 | BM25= 2 | SBERT= 3 | AdamW decouples weight decay from the adaptive gradient...
  Doc 13 | RRF=0.03200 | BM25= 3 | SBERT= 2 | Batch normalisation standardises layer inputs across th...
  Doc 07 | RRF=0.03033 | BM25= 8 | SBERT= 4 | Backpropagation applies the chain rule to compute gradi...
  Doc 14 | RRF=0.03033 | BM25= 4 | SBERT= 8 | Transfer learning fine-tunes a model pre-trained on a l...
[Step 3] Cross-encoder re-ranking (original query)...
  Doc 13 | CE=-6.7244 | Batch 

In [16]:
# Test query 3
answer_q3 = advanced_rag("what is dropout and why does it help?")


════════════════════════════════════════════════════════════
Query: what is dropout and why does it help?
────────────────────────────────────────────────────────────
[Step 1] Generating HyDE expansion...
  HyDE doc: Dropout is a regularization technique used in deep neural networks to prevent overfitting by randomly dropping out units...
[Step 2] Hybrid retrieval (BM25 + SBERT + RRF)...
  Doc 04 | RRF=0.03279 | BM25= 1 | SBERT= 1 | Dropout is a regularization technique that randomly set...
  Doc 12 | RRF=0.03151 | BM25= 5 | SBERT= 2 | AdamW decouples weight decay from the adaptive gradient...
  Doc 03 | RRF=0.03084 | BM25= 2 | SBERT= 8 | Stochastic gradient descent updates model weights by co...
  Doc 07 | RRF=0.03012 | BM25= 9 | SBERT= 4 | Backpropagation applies the chain rule to compute gradi...
  Doc 09 | RRF=0.03009 | BM25= 8 | SBERT= 5 | Convolutional neural networks apply learned filters acr...
[Step 3] Cross-encoder re-ranking (original query)...
  Doc 04 | CE=+6.4974 | Dropo

In [18]:
def naive_rag(user_query: str) -> Dict:
    sbert_model = retriever.sbert
    query_emb   = sbert_model.encode([user_query], convert_to_numpy=True)
    sims        = cosine_similarity(query_emb, retriever.corpus_embeddings)[0]
    top_idx     = int(np.argmax(sims))
    return {
        "doc_id": top_idx,
        "score":  round(float(sims[top_idx]), 4),
        "text":   CORPUS[top_idx],
    }


def advanced_rag_top_doc(user_query: str) -> Dict:
    hyp_doc    = hyde_expand(user_query)
    candidates = retriever.retrieve(hyp_doc, top_k=5)
    top_docs   = rerank(user_query, candidates, top_k=3)
    return top_docs[0]

queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is dropout and why does it help?",
]

comparison_data = []

for q in queries:
    naive_top   = naive_rag(q)
    adv_top     = advanced_rag_top_doc(q)
    different   = naive_top["doc_id"] != adv_top["doc_id"]
    comparison_data.append({
        "query":           q,
        "naive_doc_id":    naive_top["doc_id"],
        "naive_text":      naive_top["text"][:70] + "...",
        "adv_doc_id":      adv_top["doc_id"],
        "adv_text":        adv_top["text"][:70] + "...",
        "different":       different,
    })
    print(f"Query: {q}")
    print(f"  Naïve    → Doc {naive_top['doc_id']:02d}: {naive_top['text'][:65]}...")
    print(f"  Advanced → Doc {adv_top['doc_id']:02d}:  {adv_top['text'][:65]}...")
    print(f"  Different? {'YES' if different else 'NO'}\n")

Query: how do transformers encode meaning?
  Naïve    → Doc 02: Positional encodings are added to token embeddings in transformer...
  Advanced → Doc 02:  Positional encodings are added to token embeddings in transformer...
  Different? NO

Query: optimization techniques for training
  Naïve    → Doc 13: Batch normalisation standardises layer inputs across the mini-bat...
  Advanced → Doc 04:  Dropout is a regularization technique that randomly sets neuron a...
  Different? YES

Query: what is dropout and why does it help?
  Naïve    → Doc 04: Dropout is a regularization technique that randomly sets neuron a...
  Advanced → Doc 04:  Dropout is a regularization technique that randomly sets neuron a...
  Different? NO



## Comparison Table

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| `how do transformers encode meaning?` | Doc 02 | Doc 02 | NO |
| `optimization techniques for training` | Doc 13 | Doc 04 | YES |
| `what is dropout and why does it help?` | Doc 04 | Doc 04 | NO |